CharacterTextSplitter is the most basic text splitter in LangChain. It breaks a large piece of text into smaller chunks by cutting at one fixed separator you specify, and groups the resulting pieces so each chunk stays close to a maximum size you define.

In [3]:
from langchain_text_splitters import CharacterTextSplitter
# Step 1: create the splitter
splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=100,
    chunk_overlap=10,
)
# Step 2: some sample long text with paragraph breaks
long_text = """LangChain is a framework for building LLM applications.
It provides tools for prompts, chains, memory, and agents.
This makes it much easier to build complex AI apps quickly.
Developers use it to connect LLMs with external data and tools."""
# Step 3: split the text
chunks = splitter.split_text(long_text)
# Step 4: print results
print("Number of chunks:", len(chunks))
print()
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:")
    print(chunk)
    print("Length:", len(chunk))
    print("---")

Number of chunks: 1

Chunk 1:
LangChain is a framework for building LLM applications.
It provides tools for prompts, chains, memory, and agents.
This makes it much easier to build complex AI apps quickly.
Developers use it to connect LLMs with external data and tools.
Length: 238
---


Next: **RecursiveCharacterTextSplitter** — theory only, interview-ready.

**1. What it is**

`RecursiveCharacterTextSplitter` is the most commonly used splitter in LangChain, and the default recommended choice for most text. Instead of relying on just one separator like `CharacterTextSplitter`, it tries a **list of separators, in priority order**, and falls back to the next one whenever a chunk is still too big.

**2. Why it exists — fixing CharacterTextSplitter's weakness**

`CharacterTextSplitter` breaks down when a piece of text doesn't contain its one chosen separator — it has nowhere else to cut, so it can return an oversized chunk. `RecursiveCharacterTextSplitter` solves this by having a whole fallback chain of separators, so it almost always finds *some* reasonable place to cut, even in messy text.

**3. The default separator priority list**

By default, it tries, in this order:
1. `"\n\n"` — paragraph breaks (try this first)
2. `"\n"` — line breaks (if paragraphs are still too big)
3. `" "` — spaces, i.e. word boundaries (if lines are still too big)
4. `""` — individual characters (last resort, guaranteed to fit)

So it starts with the most "natural" break point (paragraphs), and only gets more aggressive/granular if it absolutely has to.

**4. How it works, conceptually — the "recursive" part**

It picks the first separator in the list and splits the text there. Then, for each resulting piece, it checks: is this still bigger than `chunk_size`? If yes, it takes THAT piece and tries splitting it again, using the NEXT separator in the list. This keeps happening — recursively — until every piece is within the size limit, or it's down to splitting by raw characters as the final fallback. This is why it's called "recursive" — it keeps reapplying the splitting logic on oversized pieces, going down the priority list each time.

**5. Same two core settings as before**

- `chunk_size`: max size per chunk.
- `chunk_overlap`: how much text from the end of one chunk is repeated at the start of the next, for context continuity.

These behave exactly the same way conceptually as in `CharacterTextSplitter` — the only difference is the smarter, multi-level splitting logic used to decide where to cut.

**6. Why it produces better chunks than CharacterTextSplitter**

Because it tries paragraph breaks first, then lines, then words, only falling to raw character cuts as an absolute last resort, it respects the natural structure of the text far more often. A sentence or idea is much less likely to get awkwardly sliced in half compared to a single-separator splitter.

**7. Customizing the separator list**

You're not stuck with the default list — you can pass your own list of separators, in your own priority order, if your text has a different natural structure (e.g., splitting code, or a transcript with custom markers).

**8. When to use it**

This is the safe, general-purpose default for almost any text-splitting task — articles, PDFs, scraped web content, chat transcripts, general documents. If asked "which splitter do you use by default," this is the answer.

**9. One-line interview answer**

"`RecursiveCharacterTextSplitter` splits text using a prioritized list of separators — paragraphs, then lines, then spaces, then characters as a last resort — recursively re-splitting any piece that's still too big using the next separator down the list. This makes it much more robust than `CharacterTextSplitter`, since it almost always finds a natural break point, and it's the default choice for most RAG and document-processing use cases."

**10. Likely follow-up**

"Why is it called 'recursive'?" → "Because whenever a split piece is still too large, it recursively applies the same splitting process to that piece, using the next separator in the priority list, until everything fits within chunk_size."

Want the same theory treatment for `TokenTextSplitter` next?

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Step 1: create the splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=10,
)

# Step 2: sample text — notice this one has NO blank lines in some parts,
# unlike the CharacterTextSplitter example
long_text = """LangChain is a framework for building LLM applications. It provides tools for prompts, chains, memory, and agents.

This makes it much easier to build complex AI apps quickly. Developers use it to connect LLMs with external data and tools.

Without a good splitting strategy, a single long paragraph like this one could end up as one oversized chunk, which is exactly the problem recursive splitting is designed to avoid."""

# Step 3: split the text
chunks = splitter.split_text(long_text)

# Step 4: print results
print("Number of chunks:", len(chunks))
print()

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:")
    print(chunk)
    print("Length:", len(chunk))
    print("---")